#part 10. 작업공간 준비 원본 불러오기, 초기상태 기록. 
Path.cwd()        # 지금 어디야?
.parent           # 한 폴더 위로
.name             # 현재 폴더 이름
/ "data" / "raw"  # 하위 경로 연결
.mkdir()          # 폴더 만들기

#customers.csv 파일의 내용을 읽어서, 파이썬이 RAM(메모리)에 표 형태로 다시 만들어 놓은 것이 DataFrame이야
원본 종이는 서랍에 그대로 있고, 책상 위에서 작업하는 거야.

디스크에 있는 파일
customers.csv
↓ 읽기
파이썬 메모리 안의 표
customers_raw(DataFrame)

CSV
= 디스크에 저장된 파일
read_csv()
= 파일 내용을 읽기
DataFrame
= 메모리에 만들어진 작업용 표
to_csv()
= DataFrame 내용을 다시 파일로 저장

#전체를 한번에 연결
CSV 파일 4개 읽기
↓
customers_raw / products_raw / orders_raw / order_items_raw 생성
↓
raw_datasets 딕셔너리에 묶음
↓
for name, df in raw_datasets.items()
↓
summarize_dataframe() 4번 호출
↓
딕셔너리 4개 생성
↓
pd.DataFrame()
↓
지금 보이는 4행짜리 표

In [3]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent.parent

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
rejected_dir = project_root / "data" / "rejected"
report_dir = project_root / "reports" / "chapter05"

for folder in [
    processed_dir,
    rejected_dir,
    report_dir,

]:

    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("프로젝트 루트:", project_root)
print("원본 폴더:", raw_dir)

프로젝트 루트: c:\stage_1\dev\llm-data-analysis-course
원본 폴더: c:\stage_1\dev\llm-data-analysis-course\data\raw


#46. 원본 데이터 불러오기 

In [5]:
customers_raw = pd.read_csv(
    raw_dir / "customers.csv"
)

products_raw = pd.read_csv(
    raw_dir / "products.csv"
)

orders_raw = pd.read_csv(
    raw_dir / "orders.csv"
)

order_items_raw = pd.read_csv(
    raw_dir / "order_items.csv"
)
print(Path.cwd())
print(project_root)
print(raw_dir)

c:\stage_1\dev\llm-data-analysis-course\notebooks\ch05
c:\stage_1\dev\llm-data-analysis-course
c:\stage_1\dev\llm-data-analysis-course\data\raw


#47. 작업용 복사본 만들기

In [6]:
customers_clean = customers_raw.copy()
products_clean = products_raw.copy()
orders_clean = orders_raw.copy()
order_items_clean = order_items_raw.copy()

#48.초기 상태 요약 함수

In [10]:
def summarize_dataframe(
    name: str,                 # 데이터셋 이름을 받음. 예: "customers"
    df: pd.DataFrame,          # 검사할 DataFrame을 받음. 예: customers_raw
    primary_key: str,          # 기본키로 검사할 컬럼 이름을 받음. 예: "customer_id"
) -> dict:                     # 결과를 딕셔너리 1개로 반환

    return {
        "dataset": name,                                   # 데이터셋 이름
        "rows": len(df),                                   # 전체 행 개수
        "columns": df.shape[1],                            # df.shape = (행 수, 열 수), [1]은 두 번째 값인 열 수
        "missing_total": int(df.isna().sum().sum()),       # DataFrame 전체 결측치 개수
        "duplicate_rows": int(df.duplicated().sum()),      # 완전히 중복된 행 개수
        "key_missing": int(df[primary_key].isna().sum()),  # 기본키 컬럼에서 비어 있는 값 개수
        "key_duplicates": int(
            df[primary_key].duplicated().sum()
        ),                                                 # 기본키 컬럼에서 중복된 값 개수
    }                                                      # 위 검사 결과 전체를 딕셔너리 1개로 반환

In [15]:
primary_keys = {
    "customers": "customer_id",          # customers 데이터셋의 기본키는 customer_id
    "products": "product_id",            # products 데이터셋의 기본키는 product_id
    "orders": "order_id",                # orders 데이터셋의 기본키는 order_id
    "order_items": "order_item_id",      # order_items 데이터셋의 기본키는 order_item_id
}

raw_datasets = {
    "customers": customers_raw,          # customers라는 이름과 customers_raw DataFrame을 연결
    "products": products_raw,            # products라는 이름과 products_raw DataFrame을 연결
    "orders": orders_raw,                # orders라는 이름과 orders_raw DataFrame을 연결
    "order_items": order_items_raw,      # order_items라는 이름과 order_items_raw DataFrame을 연결
}

before_summary = pd.DataFrame(
    [
        summarize_dataframe(             # 아까 만든 요약 함수 호출
            name,                        # 현재 데이터셋 이름 예: "customers"
            df,                          # 현재 DataFrame 예: customers_raw
            primary_keys[name],          # 현재 데이터셋의 기본키 이름 예: "customer_id"
        )
        for name, df
        in raw_datasets.items()           # raw_datasets에서 이름과 DataFrame을 하나씩 꺼내 4번 반복
    ]
)                                       # 함수가 반환한 딕셔너리 4개를 DataFrame으로 변환

before_summary                            # 최종 요약표 출력

,dataset,rows,columns,missing_total,duplicate_rows,key_missing,key_duplicates
0,customers,150,6,0,0,0,0
1,products,100,4,0,0,0,0
2,orders,300,5,0,0,0,0
3,order_items,764,5,0,0,0,0


<!-- Part 11. 문자열 공백 제거와 빈 문자열의 결측치 변환
49. 문자열 컬럼 확인 -->

In [16]:
for name, df in {
    "customers": customers_clean,
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
}.items():

    string_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    print(name, string_columns)

customers ['name', 'gender', 'city', 'signup_date']
products ['product_name', 'category']
orders ['order_date', 'payment_method', 'order_status']
order_items []


#50. 문자열 정리 함수

In [17]:

def clean_string_columns(
    df: pd.DataFrame,
) -> pd.DataFrame:
    result = df.copy()
    
    string_columns = result.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in string_columns:
        result[column] = (
            result[column]
            .astype("string")
            .str.strip()
        )

    return result

In [22]:

customers_clean = clean_string_columns(
    customers_clean
)

products_clean = clean_string_columns(
    products_clean
)

orders_clean = clean_string_columns(
    orders_clean
)

order_items_clean = clean_string_columns(
    order_items_clean
)

print(customers_clean)
print(products_clean)
print(orders_clean)
print(order_items_clean)

     customer_id name gender  age city signup_date
0              1  김수민      F   19   광주  2024-08-19
1              2  김정호      F   32   대구  2026-01-02
2              3  이경수      F   61   성남  2024-08-12
3              4  조영호      F   55   울산  2026-06-13
4              5  이예원      F   19   부산  2024-11-13
..           ...  ...    ...  ...  ...         ...
145          146  김숙자      M   61   성남  2026-02-22
146          147  이정남      M   19   부산  2025-04-13
147          148  오도현      M   29   고양  2026-08-15
148          149  김정자      M   20   부산  2024-12-19
149          150  조미영      M   40   대전  2026-02-03

[150 rows x 6 columns]
    product_id product_name category   price
0            1  전자기기 상품 001     전자기기  160000
1            2    도서 상품 002       도서   34000
2            3  전자기기 상품 003     전자기기  152000
3            4  생활용품 상품 004     생활용품   70000
4            5    식품 상품 005       식품  186000
..         ...          ...      ...     ...
95          96  생활용품 상품 096     생활용품  112000
96  